<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/C5_V4_V5_concatene_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C.5 — Concatenated Master Notebook V4 + V5

This notebook brings together the two complementary levels of non-arborescence digital diagnostics:

## Part A — V4: Reference Diagnostic Audit

V4 retains global Kron reduction on modest patches. It allows auditing:

- fine graph connectivity;
- cycle density;
- weighted incidence;
- non-local leakage produced by the Schur complement;
- energy error related to local projection;
- non-arborescent, candidate arborescent or inconclusive fragmented verdicts.

## Part B — V5: Optimized Large Patch Extension

V5 replaces Kron as the primary coarse-graining operator with the quotient projection defined in C.4:

\[G_{\ell,Z}^{\mathrm{cell4}}
\overset{\pi_{\ell,Z}^{(2)}}{\longmapsto}
\overline G_{\ell,Z}^{(2)}.

\]

Kron is then limited to a sampled local audit. This version significantly increases patch size without forming a dense global matrix.

## Methodological Hierarchy

V4 serves as the **operational reference and consistency check**.

V5 is the **recommended version for finite-size study and the large patch**.

Neither version performs an IFS \(\ell\rightarrow\ell+1\) transition. The IFS depth remains fixed; only the representation or projection level changes.

## Master Notebook Execution Parameters

Default settings execute:

- V4 in `QUICK` mode;
- V5 in `SMOKE` mode.

To run the full large patch, replace:

```python
V5_RUN_MODE = "SMOKE"
```

with:

```python
V5_RUN_MODE = "LARGE_PATCH"
```

V4 `FULL` mode is still available for comparison, but it uses global Kron reduction and can be memory-intensive.

In [ ]:
from pathlib import Path

RUN_V4 = True
V4_RUN_FULL_SCAN = False

RUN_V5 = True
V5_RUN_MODE = "LARGE_PATCH"  # "SMOKE" ou "LARGE_PATCH"

MASTER_OUTPUT_DIR = Path("./C5_V4_V5_outputs")
MASTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "RUN_V4": RUN_V4,
    "V4_RUN_FULL_SCAN": V4_RUN_FULL_SCAN,
    "RUN_V5": RUN_V5,
    "V5_RUN_MODE": V5_RUN_MODE,
    "MASTER_OUTPUT_DIR": str(MASTER_OUTPUT_DIR),
})

{'RUN_V4': True, 'V4_RUN_FULL_SCAN': False, 'RUN_V5': True, 'V5_RUN_MODE': 'LARGE_PATCH', 'MASTER_OUTPUT_DIR': 'C5_V4_V5_outputs'}


# Part A — C.5 V4

Reference diagnostic audit with global Kron reduction.

# C.5 — Partial Non-Arborescence Digital Diagnostic on a Large-Patch Cell-4 Projection

## Notebook Status

This notebook is a structural revision of V3.

It tests a **cell-4 digital proxy at fixed IFS depth**. It does not describe a new iteration of the substrate and performs no transformation of the type

\[G_\ell \longrightarrow G_{\ell+1}.

\]

The numerical reduction is denoted

\[G_{\ell,Z}^{\mathrm{cell4}}
\overset{\pi_{\ell,Z}^{(2)}}{\longmapsto}
\overline G_{\ell,Z}^{(2)},

\]

where the index \(2\) denotes a \(2\times2\) grouping, not an additional IFS level.

The notebook distinguishes:

- fixed IFS depth;
- the discretization level of the numerical proxy;
- operational resolution \(Z=\ln(\Delta\tau_\odot/t_P)\);
- Kron reduction;
- local projection applied after Kron.

## What the test establishes — and what it does not

The diagnostic combines:

- the fraction of vertices belonging to the largest connected component;
- cyclomatic number and cycle density;
- the fraction of leaves in the principal component;
- an explicit bound on weighted incidence;
- non-local leakage produced by Kron reduction;
- energy error induced by local projection;
- stability under increasing patch size.

Three families of verdicts are distinguished:

1. `pass_connected_non_tree_like`: connected and not purely tree-like proxy;
2. `candidate_tree_like`: candidate tree-like in a connected regime;
3. `inconclusive_fragmented`: regime too fragmented to conclude globally.

The test does not demonstrate the general absence of a branched-polymer phase in the IFS/PCF attractor. Nor does it construct a distinct relational network \(H_1\) capable of replacing planar adjacency in the UV regime.

In [ ]:
# ============================================================
# C.5 V4 — Diagnostic numérique partiel de non-arborescence
# sur une projection cell-4 à grand patch
#
# Garde-fous conceptuels
# ----------------------
# - ifs_depth_fixed : profondeur IFS maintenue fixe; le notebook ne réalise
#   aucune transition de Hutchinson ℓ -> ℓ+1.
# - mesh_level : niveau de discrétisation du proxy numérique uniquement.
# - Z = ln(delta_tau / t_P_cal) : coordonnée de résolution opérationnelle.
# - La projection 2x2 produit Gbar_{ℓ,Z}^{(2)}, pas G_{ℓ+1}.
# - Les unités du maillage sont normalisées dans la représentation observateur;
#   elles ne sont pas des cellules planckiennes internes.
# - Aucun résidu DSI n'est utilisé.
#
# Sorties
# -------
# - C5_results_cell4_projection_v4.csv
# - C5_summary_cell4_projection_v4.csv
# - C5_run_metadata_v4.json
# ============================================================

from __future__ import annotations

import json
import math
import os
import platform
import sys
from collections import defaultdict, deque
from dataclasses import asdict, dataclass
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

try:
    import numpy as np
    import pandas as pd
    import scipy
    from scipy.sparse import coo_matrix, csc_matrix, identity
    from scipy.sparse.linalg import splu
except Exception as exc:
    raise RuntimeError(
        "Ce notebook requiert numpy, pandas et scipy. "
        "Dans Colab, exécuter au besoin: !pip -q install numpy pandas scipy"
    ) from exc

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

Edge = Tuple[int, int]


# ============================================================
# 0) Configuration
# ============================================================

@dataclass(frozen=True)
class C5Config:
    # Statut conceptuel
    ifs_depth_fixed: int = 0
    tP_cal: float = 1.0
    base_mesh_spacing: float = 1.0

    # Balayage numérique
    mesh_levels: Tuple[int, ...] = (0, 1)
    patch_radii: Tuple[float, ...] = (24.0, 32.0, 40.0)
    delta_tau_values: Tuple[float, ...] = (1.0, 0.20, 0.08, 0.04)
    n_realizations: int = 3
    seed: int = 42

    # Fenêtre intérieure
    margin_factor: float = 3.0
    n_min_fine: int = 200
    n_min_projected: int = 80
    chi_min: float = 0.95

    # Activation relationnelle non DSI
    tau_c: float = 0.08
    p_keep_at_tau_c: float = 0.50
    p_floor: float = 0.02
    p_ceil: float = 0.995
    kappa_keep: float = 1.30
    lambda_uv: float = 5.0
    lambda_ir: float = 0.25
    kappa_lambda: float = 1.0

    # Incidence bornée sur le graphe fin
    incidence_cap: float = 24.0

    # Projection de Kron puis projection locale
    block_size: int = 2
    kron_leak_regularization: float = 1e-9
    kron_max_nodes: int = 12000
    weight_epsilon: float = 1e-12
    nonlocal_fraction_max: float = 0.35
    energy_error_max: float = 0.40
    energy_test_vectors: int = 8

    # Diagnostic de non-arborescence
    rho_loop_min: float = 0.01
    leaf_fraction_max: float = 0.75

    # Stabilité en taille
    min_patch_points: int = 2
    patch_relative_range_max: float = 0.35


# ============================================================
# 1) Outils généraux
# ============================================================

def canonical_edge(u: int, v: int) -> Edge:
    if u == v:
        raise ValueError("Une boucle propre n'est pas une arête admissible.")
    return (u, v) if u < v else (v, u)


def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def resolution_coordinate(delta_tau: float, tP_cal: float = 1.0) -> float:
    if delta_tau <= 0 or tP_cal <= 0:
        raise ValueError("delta_tau et tP_cal doivent être strictement positifs.")
    return math.log(float(delta_tau) / float(tP_cal))


def p_keep_from_resolution(
    delta_tau: float,
    tau_c: float,
    p_at_tau_c: float,
    p_floor: float,
    p_ceil: float,
    kappa: float,
) -> float:
    """Activation monotone, calibrée exactement par p(tau_c)=p_at_tau_c."""
    if not (0.0 <= p_floor < p_at_tau_c < p_ceil <= 1.0):
        raise ValueError("Il faut p_floor < p_at_tau_c < p_ceil dans [0,1].")
    ratio = (p_at_tau_c - p_floor) / (p_ceil - p_at_tau_c)
    offset = math.log(ratio)
    x = offset + kappa * math.log(max(delta_tau, 1e-15) / tau_c)
    return p_floor + (p_ceil - p_floor) * sigmoid(x)


def lambda_multiplicity_from_resolution(
    delta_tau: float,
    tau_c: float,
    lambda_uv: float,
    lambda_ir: float,
    kappa: float,
) -> float:
    """Multiplicité moyenne lisse, non oscillatoire et sans revendication DSI."""
    z = math.log(max(delta_tau, 1e-15) / tau_c)
    return lambda_ir + (lambda_uv - lambda_ir) * sigmoid(-kappa * z)


def adjacency(vertices: Iterable[int], edges: Iterable[Edge]) -> Dict[int, set]:
    vertices = set(vertices)
    out = {v: set() for v in vertices}
    for u, v in edges:
        if u in vertices and v in vertices and u != v:
            out[u].add(v)
            out[v].add(u)
    return out


def connected_components(adj: Mapping[int, set]) -> List[set]:
    seen: set = set()
    components: List[set] = []
    for start in adj:
        if start in seen:
            continue
        q = deque([start])
        seen.add(start)
        comp = {start}
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    comp.add(v)
                    q.append(v)
        components.append(comp)
    return components


def induced_subgraph(
    vertices: Iterable[int],
    edges: Iterable[Edge],
    keep: Iterable[int],
) -> Tuple[List[int], List[Edge]]:
    keep_set = set(keep)
    out_v = sorted(set(vertices).intersection(keep_set))
    out_e = [
        canonical_edge(u, v)
        for u, v in edges
        if u in keep_set and v in keep_set and u != v
    ]
    return out_v, sorted(set(out_e))


def graph_metrics(vertices: Sequence[int], edges: Sequence[Edge]) -> Dict[str, float]:
    vset = set(vertices)
    n = len(vset)
    if n == 0:
        return {
            "n": 0,
            "m": 0,
            "components": 0,
            "lcc_n": 0,
            "lcc_ratio": float("nan"),
            "b1_full": 0,
            "rho_loop_full": float("nan"),
            "b1_lcc": 0,
            "rho_loop_lcc": float("nan"),
            "degree_mean": float("nan"),
            "degree_max": float("nan"),
            "leaf_fraction_lcc": float("nan"),
        }

    e = [canonical_edge(u, v) for u, v in edges if u in vset and v in vset and u != v]
    e = sorted(set(e))
    adj = adjacency(vset, e)
    comps = connected_components(adj)
    lcc = max(comps, key=len) if comps else set()
    c = len(comps)
    m = len(e)
    b1_full = m - n + c

    e_lcc = [(u, v) for u, v in e if u in lcc and v in lcc]
    n_lcc = len(lcc)
    b1_lcc = len(e_lcc) - n_lcc + 1 if n_lcc else 0
    deg = [len(adj[v]) for v in vset]
    leaf_fraction = (
        sum(1 for v in lcc if len(adj[v]) <= 1) / n_lcc if n_lcc else float("nan")
    )

    return {
        "n": int(n),
        "m": int(m),
        "components": int(c),
        "lcc_n": int(n_lcc),
        "lcc_ratio": float(n_lcc / n),
        "b1_full": int(b1_full),
        "rho_loop_full": float(b1_full / n),
        "b1_lcc": int(b1_lcc),
        "rho_loop_lcc": float(b1_lcc / n_lcc) if n_lcc else float("nan"),
        "degree_mean": float(np.mean(deg)) if deg else float("nan"),
        "degree_max": float(np.max(deg)) if deg else float("nan"),
        "leaf_fraction_lcc": float(leaf_fraction),
    }


def weighted_incidence(
    vertices: Sequence[int],
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> Dict[str, float]:
    values = {v: 0.0 for v in vertices}
    for u, v in edges:
        key = canonical_edge(u, v)
        w = float(conductances.get(key, 0.0))
        if u in values and v in values:
            values[u] += w
            values[v] += w
    arr = np.array(list(values.values()), dtype=float)
    return {
        "incidence_mean": float(np.mean(arr)) if arr.size else float("nan"),
        "incidence_max": float(np.max(arr)) if arr.size else float("nan"),
        "incidence_q95": float(np.quantile(arr, 0.95)) if arr.size else float("nan"),
    }


# ============================================================
# 2) Proxy cell-4 normalisé
# ============================================================

def generate_lattice4_proxy(
    mesh_level: int,
    patch_radius: float,
    base_spacing: float = 1.0,
) -> Tuple[List[int], List[Edge], Dict[int, Tuple[float, float]], Dict[int, Tuple[int, int]], float]:
    """
    Réseau carré local à quatre voisins dans un disque normalisé.

    mesh_level est un indice de discrétisation numérique.
    Il ne représente pas la profondeur IFS et n'est jamais interprété comme ℓ+1.
    """
    spacing = float(base_spacing) * (2.0 ** int(mesh_level))
    if spacing <= 0 or patch_radius <= 0:
        raise ValueError("spacing et patch_radius doivent être positifs.")

    mmax = int(math.floor(patch_radius / spacing))
    coords: List[Tuple[int, int]] = []
    for i in range(-mmax, mmax + 1):
        for j in range(-mmax, mmax + 1):
            x, y = i * spacing, j * spacing
            if x * x + y * y <= patch_radius * patch_radius + 1e-12:
                coords.append((i, j))

    id_of = {ij: idx for idx, ij in enumerate(coords)}
    coord_of = {idx: ij for ij, idx in id_of.items()}
    positions = {
        idx: (ij[0] * spacing, ij[1] * spacing)
        for idx, ij in coord_of.items()
    }
    vertices = list(range(len(coords)))
    edges: List[Edge] = []
    for i, j in coords:
        u = id_of[(i, j)]
        for di, dj in ((1, 0), (0, 1)):
            nb = (i + di, j + dj)
            if nb in id_of:
                edges.append(canonical_edge(u, id_of[nb]))
    return vertices, sorted(set(edges)), positions, coord_of, spacing


def interior_vertices(
    positions: Mapping[int, Tuple[float, float]],
    patch_radius: float,
    margin: float,
) -> List[int]:
    cutoff = patch_radius - margin
    if cutoff <= 0:
        return []
    return [
        v for v, (x, y) in positions.items()
        if math.hypot(x, y) <= cutoff
    ]


def activate_weighted_relations(
    geometric_edges: Sequence[Edge],
    vertices: Sequence[int],
    rng: np.random.Generator,
    delta_tau: float,
    cfg: C5Config,
) -> Tuple[List[Edge], Dict[Edge, float], Dict[str, float]]:
    """
    Active des relations locales du proxy et leur attribue une multiplicité positive,
    avec borne d'incidence explicite sur chaque sommet.

    Cette fonction ne construit pas un réseau H1 distinct et ne prétend pas restaurer
    la connectivité lorsque le contrôle planaire se fragmente.
    """
    p_keep = p_keep_from_resolution(
        delta_tau=delta_tau,
        tau_c=cfg.tau_c,
        p_at_tau_c=cfg.p_keep_at_tau_c,
        p_floor=cfg.p_floor,
        p_ceil=cfg.p_ceil,
        kappa=cfg.kappa_keep,
    )
    lam = lambda_multiplicity_from_resolution(
        delta_tau=delta_tau,
        tau_c=cfg.tau_c,
        lambda_uv=cfg.lambda_uv,
        lambda_ir=cfg.lambda_ir,
        kappa=cfg.kappa_lambda,
    )

    incidence = {v: 0.0 for v in vertices}
    active_edges: List[Edge] = []
    conductances: Dict[Edge, float] = {}
    proposed = 0
    clipped = 0
    rejected_by_cap = 0

    order = rng.permutation(len(geometric_edges))
    for idx in order:
        u, v = geometric_edges[int(idx)]
        if rng.random() > p_keep:
            continue
        proposed += 1
        proposed_weight = float(1 + int(rng.poisson(lam)))
        remaining = min(
            cfg.incidence_cap - incidence[u],
            cfg.incidence_cap - incidence[v],
        )
        if remaining < 1.0:
            rejected_by_cap += 1
            continue
        weight = min(proposed_weight, float(math.floor(remaining)))
        if weight < proposed_weight:
            clipped += 1
        key = canonical_edge(u, v)
        active_edges.append(key)
        conductances[key] = float(weight)
        incidence[u] += weight
        incidence[v] += weight

    return sorted(set(active_edges)), conductances, {
        "Z": resolution_coordinate(delta_tau, cfg.tP_cal),
        "p_keep": float(p_keep),
        "lambda_rel": float(lam),
        "proposed_edges": int(proposed),
        "active_edges": int(len(active_edges)),
        "weights_clipped": int(clipped),
        "edges_rejected_by_incidence_cap": int(rejected_by_cap),
    }


# ============================================================
# 3) Projection 2x2 à profondeur IFS fixée
# ============================================================

def make_blocks(
    vertices: Sequence[int],
    coord_of: Mapping[int, Tuple[int, int]],
    block_size: int,
) -> Tuple[Dict[Tuple[int, int], List[int]], Dict[int, int], Dict[int, Tuple[int, int]]]:
    if block_size < 2:
        raise ValueError("block_size doit être >= 2.")
    blocks: Dict[Tuple[int, int], List[int]] = defaultdict(list)
    for u in vertices:
        i, j = coord_of[u]
        blocks[(i // block_size, j // block_size)].append(u)

    block_coords = sorted(blocks)
    block_id_of = {ij: idx for idx, ij in enumerate(block_coords)}
    block_coord_of = {idx: ij for ij, idx in block_id_of.items()}
    block_of_vertex: Dict[int, int] = {}
    for ij, members in blocks.items():
        bid = block_id_of[ij]
        for u in members:
            block_of_vertex[u] = bid
    return dict(blocks), block_of_vertex, block_coord_of


def choose_anchor_per_block(
    blocks: Mapping[Tuple[int, int], Sequence[int]],
    block_coord_of: Mapping[int, Tuple[int, int]],
) -> Tuple[List[int], Dict[int, int]]:
    block_id_of = {ij: bid for bid, ij in block_coord_of.items()}
    anchor_to_block: Dict[int, int] = {}
    for ij, members in blocks.items():
        if not members:
            continue
        anchor = min(members)
        anchor_to_block[anchor] = block_id_of[ij]
    return sorted(anchor_to_block), anchor_to_block


def local_block_edges(block_coord_of: Mapping[int, Tuple[int, int]]) -> List[Edge]:
    block_id_of = {ij: bid for bid, ij in block_coord_of.items()}
    edges: List[Edge] = []
    for bid, (i, j) in block_coord_of.items():
        for di, dj in ((1, 0), (0, 1)):
            nb = (i + di, j + dj)
            if nb in block_id_of:
                edges.append(canonical_edge(bid, block_id_of[nb]))
    return sorted(set(edges))


def reindex_graph(
    vertices: Sequence[int],
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> Tuple[List[int], List[Edge], Dict[Edge, float], Dict[int, int], Dict[int, int]]:
    old_to_new = {old: new for new, old in enumerate(sorted(vertices))}
    new_to_old = {new: old for old, new in old_to_new.items()}
    new_vertices = list(range(len(old_to_new)))
    new_edges: List[Edge] = []
    new_c: Dict[Edge, float] = {}
    for u, v in edges:
        if u not in old_to_new or v not in old_to_new:
            continue
        key_old = canonical_edge(u, v)
        key_new = canonical_edge(old_to_new[u], old_to_new[v])
        new_edges.append(key_new)
        new_c[key_new] = float(conductances[key_old])
    return new_vertices, sorted(set(new_edges)), new_c, old_to_new, new_to_old


def laplacian_from_conductances(
    n_nodes: int,
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> csc_matrix:
    diag = np.zeros(n_nodes, dtype=float)
    rows: List[int] = []
    cols: List[int] = []
    data: List[float] = []
    for u, v in edges:
        w = float(conductances[canonical_edge(u, v)])
        diag[u] += w
        diag[v] += w
        rows.extend((u, v))
        cols.extend((v, u))
        data.extend((-w, -w))
    rows.extend(range(n_nodes))
    cols.extend(range(n_nodes))
    data.extend(diag.tolist())
    out = coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes)).tocsc()
    out.sum_duplicates()
    return out


def kron_reduction(
    laplacian: csc_matrix,
    keep_indices: Sequence[int],
    leak_regularization: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = laplacian.shape[0]
    keep = np.array(sorted(set(int(i) for i in keep_indices)), dtype=int)
    if keep.size == 0:
        raise ValueError("Aucun degré conservé pour la réduction de Kron.")
    mask = np.ones(n, dtype=bool)
    mask[keep] = False
    eliminated = np.where(mask)[0]

    l_kk = laplacian[keep[:, None], keep]
    if eliminated.size == 0:
        return keep, eliminated, l_kk.toarray()

    l_ki = laplacian[keep[:, None], eliminated]
    l_ik = laplacian[eliminated[:, None], keep]
    l_ii = laplacian[eliminated[:, None], eliminated]

    diagonal = np.asarray(l_ii.diagonal(), dtype=float)
    positive = diagonal[diagonal > 0]
    scale = float(np.mean(positive)) if positive.size else 1.0
    eps = max(float(leak_regularization) * scale, 1e-15)
    l_ii_reg = (l_ii + eps * identity(eliminated.size, format="csc")).tocsc()

    lu = splu(l_ii_reg)
    rhs = l_ik.toarray()
    solution = lu.solve(rhs)
    l_eff = l_kk.toarray() - (l_ki @ solution)
    l_eff = 0.5 * (l_eff + l_eff.T)
    return keep, eliminated, l_eff


def local_projection_from_kron(
    l_eff: np.ndarray,
    coarse_ids_in_eff_order: Sequence[int],
    allowed_local_edges: Sequence[Edge],
    weight_epsilon: float,
) -> Tuple[List[Edge], Dict[Edge, float], Dict[str, float], np.ndarray]:
    idx_of_block = {bid: idx for idx, bid in enumerate(coarse_ids_in_eff_order)}
    total_weight = 0.0
    local_weight = 0.0

    k = l_eff.shape[0]
    for a in range(k):
        for b in range(a + 1, k):
            w = float(-l_eff[a, b])
            if w > weight_epsilon:
                total_weight += w

    local_edges: List[Edge] = []
    local_c: Dict[Edge, float] = {}
    l_local = np.zeros_like(l_eff, dtype=float)

    for a, b in allowed_local_edges:
        ia = idx_of_block.get(a)
        ib = idx_of_block.get(b)
        if ia is None or ib is None:
            continue
        w = float(-l_eff[ia, ib])
        if w <= weight_epsilon:
            continue
        key = canonical_edge(a, b)
        local_edges.append(key)
        local_c[key] = w
        local_weight += w
        l_local[ia, ia] += w
        l_local[ib, ib] += w
        l_local[ia, ib] -= w
        l_local[ib, ia] -= w

    nonlocal_weight = max(total_weight - local_weight, 0.0)
    nonlocal_fraction = nonlocal_weight / total_weight if total_weight > 0 else float("nan")
    return sorted(set(local_edges)), local_c, {
        "total_effective_offdiag_weight": float(total_weight),
        "local_retained_weight": float(local_weight),
        "nonlocal_removed_weight": float(nonlocal_weight),
        "nonlocal_fraction": float(nonlocal_fraction),
    }, l_local


def energy_projection_error(
    l_eff: np.ndarray,
    l_local: np.ndarray,
    rng: np.random.Generator,
    n_vectors: int,
) -> Dict[str, float]:
    if l_eff.size == 0:
        return {"energy_error_mean": float("nan"), "energy_error_max": float("nan")}
    errors: List[float] = []
    n = l_eff.shape[0]
    for _ in range(max(1, int(n_vectors))):
        x = rng.normal(size=n)
        x = x - np.mean(x)
        e_eff = float(x @ l_eff @ x)
        e_loc = float(x @ l_local @ x)
        denom = max(abs(e_eff), 1e-12)
        errors.append(abs(e_eff - e_loc) / denom)
    return {
        "energy_error_mean": float(np.mean(errors)),
        "energy_error_max": float(np.max(errors)),
    }


# ============================================================
# 4) Verdicts bornés
# ============================================================

def classify_realization(row: Mapping[str, float], cfg: C5Config) -> str:
    if int(row["fine_quality_pass"]) == 0:
        return "inconclusive_fragmented"
    if row.get("projection_status") != "completed":
        return "inconclusive_projection_unavailable"
    if int(row["projected_quality_pass"]) == 0:
        return "inconclusive_fragmented_after_projection"
    if int(row["incidence_pass"]) == 0:
        return "fail_incidence"
    if int(row["local_projection_pass"]) == 0:
        return "fail_nonlocal_projection"

    rho_f = float(row["rho_loop_lcc"])
    rho_p = float(row["rho_loop_lcc_projected"])
    leaf_f = float(row["leaf_fraction_lcc"])
    leaf_p = float(row["leaf_fraction_lcc_projected"])

    if (
        rho_f >= cfg.rho_loop_min
        and rho_p >= cfg.rho_loop_min
        and leaf_f <= cfg.leaf_fraction_max
        and leaf_p <= cfg.leaf_fraction_max
    ):
        return "pass_connected_non_tree_like"

    if (
        rho_f < cfg.rho_loop_min
        and rho_p < cfg.rho_loop_min
        and leaf_f > cfg.leaf_fraction_max
        and leaf_p > cfg.leaf_fraction_max
    ):
        return "candidate_tree_like"

    return "inconclusive_mixed_cycle_diagnostics"


def relative_range(values: Sequence[float]) -> float:
    arr = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size < 2:
        return float("nan")
    denom = max(abs(float(np.mean(arr))), 1e-12)
    return float((np.max(arr) - np.min(arr)) / denom)


def summarize_results(df: pd.DataFrame, cfg: C5Config) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    group_cols = ["delta_tau", "mesh_level"]
    for keys, group in df.groupby(group_cols, dropna=False):
        delta_tau, mesh_level = keys
        group = group.sort_values(["patch_radius", "realization"])

        # Agrégation par taille de patch sur les réalisations.
        by_radius = (
            group.groupby("patch_radius", as_index=False)
            .agg(
                fine_quality_rate=("fine_quality_pass", "mean"),
                projected_quality_rate=("projected_quality_pass", "mean"),
                incidence_pass_rate=("incidence_pass", "mean"),
                local_projection_pass_rate=("local_projection_pass", "mean"),
                rho_fine_mean=("rho_loop_lcc", "mean"),
                rho_projected_mean=("rho_loop_lcc_projected", "mean"),
                leaf_fine_mean=("leaf_fraction_lcc", "mean"),
                leaf_projected_mean=("leaf_fraction_lcc_projected", "mean"),
                nonlocal_fraction_mean=("nonlocal_fraction", "mean"),
                energy_error_mean=("energy_error_mean", "mean"),
            )
        )

        admissible = by_radius[
            (by_radius["fine_quality_rate"] >= 0.5)
            & (by_radius["projected_quality_rate"] >= 0.5)
            & (by_radius["incidence_pass_rate"] >= 1.0)
            & (by_radius["local_projection_pass_rate"] >= 0.5)
        ].copy()

        last_radius = float(group["patch_radius"].max())
        last = group[group["patch_radius"] == last_radius]
        verdict_counts = last["verdict"].value_counts(dropna=False).to_dict()
        dominant_verdict = (
            str(last["verdict"].mode().iloc[0])
            if not last.empty
            else "inconclusive_no_data"
        )

        if len(admissible) >= cfg.min_patch_points:
            rho_range_f = relative_range(admissible["rho_fine_mean"].tolist())
            rho_range_p = relative_range(admissible["rho_projected_mean"].tolist())
            patch_stable = int(
                rho_range_f <= cfg.patch_relative_range_max
                and rho_range_p <= cfg.patch_relative_range_max
            )
        else:
            rho_range_f = float("nan")
            rho_range_p = float("nan")
            patch_stable = 0

        if dominant_verdict == "pass_connected_non_tree_like" and patch_stable:
            summary_verdict = "pass_large_patch_non_tree_like"
        elif dominant_verdict == "pass_connected_non_tree_like":
            summary_verdict = "provisional_pass_patch_stability_not_established"
        elif dominant_verdict.startswith("inconclusive"):
            summary_verdict = dominant_verdict
        else:
            summary_verdict = dominant_verdict

        rows.append({
            "delta_tau": float(delta_tau),
            "Z": float(resolution_coordinate(float(delta_tau), cfg.tP_cal)),
            "mesh_level": int(mesh_level),
            "patch_radius_max": last_radius,
            "n_realizations_at_max_radius": int(len(last)),
            "dominant_realization_verdict_at_max_radius": dominant_verdict,
            "verdict_counts_at_max_radius": json.dumps(verdict_counts, sort_keys=True),
            "admissible_patch_points": int(len(admissible)),
            "rho_relative_range_fine": rho_range_f,
            "rho_relative_range_projected": rho_range_p,
            "large_patch_stability_pass": int(patch_stable),
            "summary_verdict": summary_verdict,
        })

    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


# ============================================================
# 5) Runner
# ============================================================

def run_c5_v4(cfg: C5Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows: List[Dict[str, object]] = []

    for delta_tau in cfg.delta_tau_values:
        for patch_radius in cfg.patch_radii:
            for mesh_level in cfg.mesh_levels:
                for realization in range(cfg.n_realizations):
                    local_seed = (
                        int(cfg.seed)
                        + int(round(delta_tau * 1e6))
                        + 100003 * int(mesh_level)
                        + 1009 * int(round(patch_radius))
                        + 7919 * int(realization)
                    )
                    rng = np.random.default_rng(local_seed)

                    vertices, geo_edges, positions, coord_of, spacing = generate_lattice4_proxy(
                        mesh_level=mesh_level,
                        patch_radius=patch_radius,
                        base_spacing=cfg.base_mesh_spacing,
                    )

                    active_edges, conductances, activation = activate_weighted_relations(
                        geometric_edges=geo_edges,
                        vertices=vertices,
                        rng=rng,
                        delta_tau=delta_tau,
                        cfg=cfg,
                    )

                    margin = cfg.margin_factor * spacing
                    keep_interior = interior_vertices(positions, patch_radius, margin)
                    v_int, e_int = induced_subgraph(vertices, active_edges, keep_interior)
                    c_int = {
                        edge: conductances[edge]
                        for edge in e_int
                        if edge in conductances
                    }

                    fine_metrics = graph_metrics(v_int, e_int)
                    fine_incidence = weighted_incidence(v_int, e_int, c_int)
                    fine_quality_pass = int(
                        fine_metrics["n"] >= cfg.n_min_fine
                        and fine_metrics["lcc_ratio"] >= cfg.chi_min
                    )
                    incidence_pass = int(
                        np.isfinite(fine_incidence["incidence_max"])
                        and fine_incidence["incidence_max"] <= cfg.incidence_cap + 1e-9
                    )

                    base_row: Dict[str, object] = {
                        "ifs_depth_fixed": int(cfg.ifs_depth_fixed),
                        "delta_tau": float(delta_tau),
                        "Z": float(activation["Z"]),
                        "mesh_level": int(mesh_level),
                        "mesh_spacing_normalized": float(spacing),
                        "patch_radius": float(patch_radius),
                        "realization": int(realization),
                        "seed_used": int(local_seed),
                        "p_keep": float(activation["p_keep"]),
                        "lambda_rel": float(activation["lambda_rel"]),
                        "weights_clipped": int(activation["weights_clipped"]),
                        "edges_rejected_by_incidence_cap": int(
                            activation["edges_rejected_by_incidence_cap"]
                        ),
                        "n_fine": int(fine_metrics["n"]),
                        "m_fine": int(fine_metrics["m"]),
                        "components_fine": int(fine_metrics["components"]),
                        "lcc_ratio": float(fine_metrics["lcc_ratio"]),
                        "rho_loop_full": float(fine_metrics["rho_loop_full"]),
                        "rho_loop_lcc": float(fine_metrics["rho_loop_lcc"]),
                        "leaf_fraction_lcc": float(fine_metrics["leaf_fraction_lcc"]),
                        "degree_mean": float(fine_metrics["degree_mean"]),
                        "degree_max": float(fine_metrics["degree_max"]),
                        "incidence_mean": float(fine_incidence["incidence_mean"]),
                        "incidence_q95": float(fine_incidence["incidence_q95"]),
                        "incidence_max": float(fine_incidence["incidence_max"]),
                        "fine_quality_pass": int(fine_quality_pass),
                        "incidence_pass": int(incidence_pass),
                        "projection_status": "not_started",
                        "n_projected": np.nan,
                        "m_projected": np.nan,
                        "lcc_ratio_projected": np.nan,
                        "rho_loop_full_projected": np.nan,
                        "rho_loop_lcc_projected": np.nan,
                        "leaf_fraction_lcc_projected": np.nan,
                        "degree_max_projected": np.nan,
                        "incidence_max_projected": np.nan,
                        "projected_quality_pass": 0,
                        "total_effective_offdiag_weight": np.nan,
                        "local_retained_weight": np.nan,
                        "nonlocal_removed_weight": np.nan,
                        "nonlocal_fraction": np.nan,
                        "energy_error_mean": np.nan,
                        "energy_error_max": np.nan,
                        "local_projection_pass": 0,
                    }

                    # En fragmentation forte, le diagnostic global est non concluant:
                    # la projection de Kron n'est pas utilisée pour fabriquer une conclusion.
                    if fine_quality_pass == 0:
                        base_row["projection_status"] = "skipped_fragmented"
                        base_row["verdict"] = classify_realization(base_row, cfg)
                        rows.append(base_row)
                        continue

                    if len(v_int) > cfg.kron_max_nodes:
                        base_row["projection_status"] = "skipped_size_limit"
                        base_row["verdict"] = classify_realization(base_row, cfg)
                        rows.append(base_row)
                        continue

                    # Réindexation du graphe intérieur pour la matrice.
                    v_new, e_new, c_new, old_to_new, new_to_old = reindex_graph(
                        v_int, e_int, c_int
                    )
                    coord_new = {old_to_new[old]: coord_of[old] for old in v_int}

                    blocks, block_of_vertex, block_coord_of = make_blocks(
                        v_new, coord_new, cfg.block_size
                    )
                    keep_anchors, anchor_to_block = choose_anchor_per_block(
                        blocks, block_coord_of
                    )

                    lap = laplacian_from_conductances(len(v_new), e_new, c_new)
                    try:
                        keep_sorted, eliminated, l_eff = kron_reduction(
                            laplacian=lap,
                            keep_indices=keep_anchors,
                            leak_regularization=cfg.kron_leak_regularization,
                        )
                    except Exception as exc:
                        base_row["projection_status"] = f"failed:{type(exc).__name__}"
                        base_row["verdict"] = classify_realization(base_row, cfg)
                        rows.append(base_row)
                        continue

                    coarse_ids_order = [anchor_to_block[int(anchor)] for anchor in keep_sorted]
                    allowed_edges = local_block_edges(block_coord_of)
                    e_proj, c_proj, leakage, l_local = local_projection_from_kron(
                        l_eff=l_eff,
                        coarse_ids_in_eff_order=coarse_ids_order,
                        allowed_local_edges=allowed_edges,
                        weight_epsilon=cfg.weight_epsilon,
                    )
                    energy = energy_projection_error(
                        l_eff=l_eff,
                        l_local=l_local,
                        rng=rng,
                        n_vectors=cfg.energy_test_vectors,
                    )

                    v_proj = sorted(set(coarse_ids_order))
                    # Restreindre les arêtes aux blocs effectivement conservés.
                    v_proj, e_proj = induced_subgraph(v_proj, e_proj, v_proj)
                    c_proj = {edge: c_proj[edge] for edge in e_proj if edge in c_proj}
                    projected_metrics = graph_metrics(v_proj, e_proj)
                    projected_incidence = weighted_incidence(v_proj, e_proj, c_proj)

                    projected_quality_pass = int(
                        projected_metrics["n"] >= cfg.n_min_projected
                        and projected_metrics["lcc_ratio"] >= cfg.chi_min
                    )
                    local_projection_pass = int(
                        np.isfinite(leakage["nonlocal_fraction"])
                        and leakage["nonlocal_fraction"] <= cfg.nonlocal_fraction_max
                        and np.isfinite(energy["energy_error_max"])
                        and energy["energy_error_max"] <= cfg.energy_error_max
                    )

                    base_row.update({
                        "projection_status": "completed",
                        "n_projected": int(projected_metrics["n"]),
                        "m_projected": int(projected_metrics["m"]),
                        "lcc_ratio_projected": float(projected_metrics["lcc_ratio"]),
                        "rho_loop_full_projected": float(projected_metrics["rho_loop_full"]),
                        "rho_loop_lcc_projected": float(projected_metrics["rho_loop_lcc"]),
                        "leaf_fraction_lcc_projected": float(
                            projected_metrics["leaf_fraction_lcc"]
                        ),
                        "degree_max_projected": float(projected_metrics["degree_max"]),
                        "incidence_max_projected": float(
                            projected_incidence["incidence_max"]
                        ),
                        "projected_quality_pass": int(projected_quality_pass),
                        "total_effective_offdiag_weight": float(
                            leakage["total_effective_offdiag_weight"]
                        ),
                        "local_retained_weight": float(leakage["local_retained_weight"]),
                        "nonlocal_removed_weight": float(leakage["nonlocal_removed_weight"]),
                        "nonlocal_fraction": float(leakage["nonlocal_fraction"]),
                        "energy_error_mean": float(energy["energy_error_mean"]),
                        "energy_error_max": float(energy["energy_error_max"]),
                        "local_projection_pass": int(local_projection_pass),
                    })
                    base_row["verdict"] = classify_realization(base_row, cfg)
                    rows.append(base_row)

    results = pd.DataFrame(rows)
    summary = summarize_results(results, cfg)
    return results, summary


# ============================================================
# 6) Autotests
# ============================================================

def run_autotests() -> None:
    cfg = C5Config(
        mesh_levels=(1,),
        patch_radii=(12.0,),
        delta_tau_values=(0.20,),
        n_realizations=1,
        n_min_fine=20,
        n_min_projected=5,
        kron_max_nodes=2000,
        energy_test_vectors=3,
    )

    p = p_keep_from_resolution(
        delta_tau=cfg.tau_c,
        tau_c=cfg.tau_c,
        p_at_tau_c=cfg.p_keep_at_tau_c,
        p_floor=cfg.p_floor,
        p_ceil=cfg.p_ceil,
        kappa=cfg.kappa_keep,
    )
    assert abs(p - cfg.p_keep_at_tau_c) < 1e-12

    results, summary = run_c5_v4(cfg)
    assert len(results) == 1
    assert "ell" not in results.columns
    assert "good_next" not in results.columns
    assert float(results["incidence_max"].iloc[0]) <= cfg.incidence_cap + 1e-9
    assert results["verdict"].iloc[0] in {
        "pass_connected_non_tree_like",
        "candidate_tree_like",
        "inconclusive_fragmented",
        "inconclusive_projection_unavailable",
        "inconclusive_fragmented_after_projection",
        "inconclusive_mixed_cycle_diagnostics",
        "fail_incidence",
        "fail_nonlocal_projection",
    }
    assert len(summary) == 1
    print("Autotests C.5 V4: OK")

## Execution

The `QUICK` mode is activated by default to quickly verify the pipeline and exports.

To run the scan with a larger patch, replace:

```python
RUN_FULL_SCAN = False
```

with:

```python
RUN_FULL_SCAN = True
```

Kron reduction forms a dense Schur complement; therefore, full mode can be memory-intensive.

In [ ]:
# ============================================================
# 7) Exécution principale
# ============================================================

run_autotests()

# Le mode rapide vérifie le pipeline sur des tailles modestes.
# Le mode complet prépare la vérification à grand patch; il peut être coûteux
# car la réduction de Kron produit un complément de Schur dense.
RUN_FULL_SCAN = V4_RUN_FULL_SCAN

QUICK_CONFIG = C5Config(
    ifs_depth_fixed=0,
    mesh_levels=(1,),
    patch_radii=(16.0, 20.0, 24.0),
    delta_tau_values=(1.0, 0.20, 0.08, 0.04),
    n_realizations=2,
    seed=42,
    n_min_fine=40,
    n_min_projected=12,
    kron_max_nodes=4000,
    incidence_cap=24.0,
    nonlocal_fraction_max=0.35,
    energy_error_max=0.40,
)

FULL_CONFIG = C5Config(
    ifs_depth_fixed=0,
    mesh_levels=(0, 1),
    patch_radii=(24.0, 32.0, 40.0),
    delta_tau_values=(1.0, 0.20, 0.08, 0.04),
    n_realizations=3,
    seed=42,
    incidence_cap=24.0,
    nonlocal_fraction_max=0.35,
    energy_error_max=0.40,
    kron_max_nodes=12000,
)

CONFIG = FULL_CONFIG if RUN_FULL_SCAN else QUICK_CONFIG
print("Mode sélectionné:", "FULL" if RUN_FULL_SCAN else "QUICK")
print("Attention: activer FULL uniquement avec une mémoire suffisante.")

results_v4, summary_v4 = run_c5_v4(CONFIG) if RUN_V4 else (pd.DataFrame(), pd.DataFrame())

display_columns = [
    "delta_tau", "Z", "mesh_level", "patch_radius", "realization",
    "p_keep", "n_fine", "lcc_ratio", "rho_loop_lcc",
    "incidence_max", "projection_status", "n_projected",
    "lcc_ratio_projected", "rho_loop_lcc_projected",
    "nonlocal_fraction", "energy_error_max", "verdict",
]
if RUN_V4:
    display(results_v4[display_columns].head(30))
    display(summary_v4)

results_path = MASTER_OUTPUT_DIR / "C5_results_cell4_projection_v4.csv"
summary_path = MASTER_OUTPUT_DIR / "C5_summary_cell4_projection_v4.csv"
metadata_path = MASTER_OUTPUT_DIR / "C5_run_metadata_v4.json"

if RUN_V4:
    results_v4.to_csv(results_path, index=False)
    summary_v4.to_csv(summary_path, index=False)

metadata = {
    "test_name": "C5_partial_non_arborescence_cell4_projection_v4",
    "scope": (
        "Diagnostic partiel sur proxy cell-4 à profondeur IFS fixe; "
        "aucune revendication de topologie globale du substrat IFS/PCF."
    ),
    "coarse_graining_semantics": (
        "Projection 2x2 G_{ell,Z} -> Gbar_{ell,Z}^{(2)}; "
        "aucune transition ell -> ell+1."
    ),
    "dsi_used": False,
    "run_mode": "FULL" if RUN_FULL_SCAN else "QUICK",
    "config": asdict(CONFIG),
    "versions": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
    },
}
if RUN_V4:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

if RUN_V4:
    print("Exports V4 générés:")
    print(" -", results_path)
    print(" -", summary_path)
    print(" -", metadata_path)

Autotests C.5 V4: OK
Mode sélectionné: QUICK
Attention: activer FULL uniquement avec une mémoire suffisante.


,delta_tau,Z,mesh_level,patch_radius,realization,p_keep,n_fine,lcc_ratio,rho_loop_lcc,incidence_max,projection_status,n_projected,lcc_ratio_projected,rho_loop_lcc_projected,nonlocal_fraction,energy_error_max,verdict
0,1.00,0.000000,1,16.0,0,0.958700,81,1.000000,0.703704,11.0,completed,26.0,1.000000,0.538462,0.192120,0.232103,pass_connected_non_tree_like
1,1.00,0.000000,1,16.0,1,0.958700,81,1.000000,0.654321,9.0,completed,26.0,1.000000,0.538462,0.163822,0.218319,pass_connected_non_tree_like
2,1.00,0.000000,1,20.0,0,0.958700,149,1.000000,0.718121,12.0,completed,45.0,1.000000,0.666667,0.196229,0.260442,pass_connected_non_tree_like
3,1.00,0.000000,1,20.0,1,0.958700,149,1.000000,0.751678,12.0,completed,45.0,1.000000,0.644444,0.222403,0.299921,pass_connected_non_tree_like
4,1.00,0.000000,1,24.0,0,0.958700,253,1.000000,0.766798,10.0,completed,73.0,1.000000,0.739726,0.242813,0.265293,pass_connected_non_tree_like
5,1.00,0.000000,1,24.0,1,0.958700,253,1.000000,0.786561,11.0,completed,73.0,1.000000,0.739726,0.248787,0.284510,pass_connected_non_tree_like
6,0.20,-1.609438,1,16.0,0,0.762371,81,0.950617,0.337662,14.0,completed,26.0,0.923077,0.416667,0.190562,0.335540,inconclusive_fragmented_after_projection
7,0.20,-1.609438,1,16.0,1,0.762371,81,0.962963,0.346154,15.0,completed,26.0,0.923077,0.500000,0.173699,0.203465,inconclusive_fragmented_after_projection
8,0.20,-1.609438,1,20.0,0,0.762371,149,0.993289,0.425676,17.0,completed,45.0,0.977778,0.613636,0.188894,0.218823,pass_connected_non_tree_like
9,0.20,-1.609438,1,20.0,1,0.762371,149,0.986577,0.353741,19.0,completed,45.0,0.955556,0.465116,0.146095,0.285306,pass_connected_non_tree_like


,delta_tau,Z,mesh_level,patch_radius_max,n_realizations_at_max_radius,dominant_realization_verdict_at_max_radius,verdict_counts_at_max_radius,admissible_patch_points,rho_relative_range_fine,rho_relative_range_projected,large_patch_stability_pass,summary_verdict
0,0.04,-3.218876,1,24.0,2,inconclusive_fragmented,"{""inconclusive_fragmented"": 2}",0,NaN,NaN,0,inconclusive_fragmented
1,0.08,-2.525729,1,24.0,2,inconclusive_fragmented,"{""inconclusive_fragmented"": 2}",0,NaN,NaN,0,inconclusive_fragmented
2,0.20,-1.609438,1,24.0,2,pass_connected_non_tree_like,"{""pass_connected_non_tree_like"": 2}",2,0.032117,0.171377,1,pass_large_patch_non_tree_like
3,1.00,0.000000,1,24.0,2,pass_connected_non_tree_like,"{""pass_connected_non_tree_like"": 2}",3,0.133755,0.312241,1,pass_large_patch_non_tree_like


Exports V4 générés:
 - C5_V4_V5_outputs/C5_results_cell4_projection_v4.csv
 - C5_V4_V5_outputs/C5_summary_cell4_projection_v4.csv
 - C5_V4_V5_outputs/C5_run_metadata_v4.json


## Reading Exports

The notebook generates:

- `C5_results_cell4_projection_v4.csv`: results by realization;
- `C5_summary_cell4_projection_v4.csv`: summary by resolution and mesh level;
- `C5_run_metadata_v4.json`: configuration, versions, and conceptual guardrails.

A fragmented UV verdict must remain **inconclusive**. It should not be rephrased as evidence that an overlap network already restores connectivity.

Locality after Kron reduction is also precisely qualified: it results from an audited local projection, not an automatic preservation of locality by the Schur complement.

# Part B — C.5 V5

Optimized extension for the large patch.

# C.5 — Partial Non-Arborescence Digital Diagnostic
## Optimized Large Patch Version V5

This version replaces global Kron reduction with the quotient projection defined in C.4:

\[G_{\ell,Z}^{\mathrm{cell4}}
\overset{\pi_{\ell,Z}^{(2)}}{\longmapsto}
\overline G_{\ell,Z}^{(2)}.

\]

The IFS depth \(\ell\) remains fixed. The \(2\times2\) grouping constitutes a projection of the numerical representation, not an additional iteration of the substrate.

Block conductances are obtained by summing the fine conductances traversing their preimages. For block-constant functions, this projection exactly preserves the energy form:

\[\mathcal E_{\mathrm{fine}}(f\circ\pi_{\ell,Z}^{(2)})
=
\overline{\mathcal E}_{\ell,Z}(f).

\]

This construction avoids the dense matrix produced by global Kron reduction and allows processing patches containing tens of thousands of vertices.

## Modifications compared to V4

- large patch scan radii: \(R=32,48,64,96\);
- eight realizations per configuration;
- primary projection with approximately linear time and memory complexity in the number of edges;
- no dense global matrix;
- CSV checkpoint after each configuration group;
- automatic resume if the results file already exists;
- explicit control of quotient graph locality;
- numerical verification of the projection's energetic identity;
- Kron reduction retained only as a **sampled local audit** on small windows;
- UV verdict always classified as inconclusive when the proxy is fragmented.

The full mode includes 128 configurations:

\[4\ \text{resolutions}
\times
4\ \text{radii}
\times
8\ \text{realizations}.

\]

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import sys
from collections import defaultdict, deque
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import coo_matrix, csc_matrix, identity
from scipy.sparse.linalg import splu

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

Edge = Tuple[int, int]


# ============================================================
# 0) Configuration
# ============================================================

@dataclass(frozen=True)
class C5LargePatchConfig:
    # Statut conceptuel
    ifs_depth_fixed: int = 0
    tP_cal: float = 1.0
    base_mesh_spacing: float = 1.0
    mesh_levels: Tuple[int, ...] = (0,)

    # Grand patch
    patch_radii: Tuple[float, ...] = (32.0, 48.0, 64.0, 96.0)
    delta_tau_values: Tuple[float, ...] = (1.0, 0.20, 0.08, 0.04)
    n_realizations: int = 8
    seed: int = 42

    # Fenêtre intérieure / qualité
    margin_factor: float = 3.0
    n_min_fine: int = 1500
    n_min_projected: int = 350
    chi_min: float = 0.95

    # Activation relationnelle non DSI
    tau_c: float = 0.08
    p_keep_at_tau_c: float = 0.50
    p_floor: float = 0.02
    p_ceil: float = 0.995
    kappa_keep: float = 1.30
    lambda_uv: float = 5.0
    lambda_ir: float = 0.25
    kappa_lambda: float = 1.0

    # Incidence bornée
    incidence_cap: float = 24.0

    # Projection quotient par blocs, à profondeur IFS fixée
    block_size: int = 2
    energy_identity_tol: float = 1e-11
    energy_test_vectors: int = 6

    # Diagnostic de non-arborescence
    rho_loop_min: float = 0.01
    leaf_fraction_max: float = 0.75

    # Stabilité en taille
    min_patch_points: int = 3
    patch_relative_range_max: float = 0.25

    # Audit local optionnel de Kron — jamais appliqué globalement
    run_local_kron_audit: bool = True
    kron_audit_samples: int = 4
    kron_audit_block_radius: int = 2
    kron_audit_max_nodes: int = 700
    kron_leak_regularization: float = 1e-9
    weight_epsilon: float = 1e-12
    kron_nonlocal_fraction_max: float = 0.35
    kron_energy_error_max: float = 0.40
    kron_energy_test_vectors: int = 4

    # Exports / reprise
    results_csv: str = "C5_results_large_patch_v5.csv"
    summary_csv: str = "C5_summary_large_patch_v5.csv"
    metadata_json: str = "C5_run_metadata_large_patch_v5.json"
    resume: bool = True


def config_for_mode(mode: str) -> C5LargePatchConfig:
    mode = str(mode).upper().strip()
    base = C5LargePatchConfig()
    if mode == "LARGE_PATCH":
        return base
    if mode == "SMOKE":
        return replace(
            base,
            patch_radii=(16.0, 24.0),
            delta_tau_values=(1.0, 0.20, 0.08),
            n_realizations=1,
            n_min_fine=70,
            n_min_projected=20,
            min_patch_points=2,
            kron_audit_samples=2,
            results_csv="C5_results_large_patch_v5_smoke.csv",
            summary_csv="C5_summary_large_patch_v5_smoke.csv",
            metadata_json="C5_run_metadata_large_patch_v5_smoke.json",
            resume=False,
        )
    raise ValueError("mode doit être 'SMOKE' ou 'LARGE_PATCH'.")


# ============================================================
# 1) Outils de graphe
# ============================================================

def canonical_edge(u: int, v: int) -> Edge:
    if u == v:
        raise ValueError("Une boucle propre n'est pas une arête admissible.")
    return (u, v) if u < v else (v, u)


def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def resolution_coordinate(delta_tau: float, tP_cal: float = 1.0) -> float:
    if delta_tau <= 0 or tP_cal <= 0:
        raise ValueError("delta_tau et tP_cal doivent être strictement positifs.")
    return math.log(float(delta_tau) / float(tP_cal))


def p_keep_from_resolution(
    delta_tau: float,
    tau_c: float,
    p_at_tau_c: float,
    p_floor: float,
    p_ceil: float,
    kappa: float,
) -> float:
    if not (0.0 <= p_floor < p_at_tau_c < p_ceil <= 1.0):
        raise ValueError("Il faut p_floor < p_at_tau_c < p_ceil dans [0,1].")
    ratio = (p_at_tau_c - p_floor) / (p_ceil - p_at_tau_c)
    offset = math.log(ratio)
    x = offset + kappa * math.log(max(delta_tau, 1e-15) / tau_c)
    return p_floor + (p_ceil - p_floor) * sigmoid(x)


def lambda_multiplicity_from_resolution(
    delta_tau: float,
    tau_c: float,
    lambda_uv: float,
    lambda_ir: float,
    kappa: float,
) -> float:
    z = math.log(max(delta_tau, 1e-15) / tau_c)
    return lambda_ir + (lambda_uv - lambda_ir) * sigmoid(-kappa * z)


def adjacency(vertices: Iterable[int], edges: Iterable[Edge]) -> Dict[int, set]:
    vertices = set(vertices)
    out = {v: set() for v in vertices}
    for u, v in edges:
        if u in vertices and v in vertices and u != v:
            out[u].add(v)
            out[v].add(u)
    return out


def connected_components(adj: Mapping[int, set]) -> List[set]:
    seen: set = set()
    components: List[set] = []
    for start in adj:
        if start in seen:
            continue
        q = deque([start])
        seen.add(start)
        comp = {start}
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    comp.add(v)
                    q.append(v)
        components.append(comp)
    return components


def graph_metrics(vertices: Sequence[int], edges: Sequence[Edge]) -> Dict[str, float]:
    vset = set(vertices)
    n = len(vset)
    if n == 0:
        return {
            "n": 0, "m": 0, "components": 0, "lcc_n": 0,
            "lcc_ratio": float("nan"), "b1_full": 0,
            "rho_loop_full": float("nan"), "b1_lcc": 0,
            "rho_loop_lcc": float("nan"), "degree_mean": float("nan"),
            "degree_max": float("nan"), "leaf_fraction_lcc": float("nan"),
        }

    e = sorted(set(canonical_edge(u, v) for u, v in edges if u in vset and v in vset and u != v))
    adj = adjacency(vset, e)
    comps = connected_components(adj)
    lcc = max(comps, key=len) if comps else set()
    c = len(comps)
    m = len(e)
    b1_full = m - n + c
    e_lcc = [(u, v) for u, v in e if u in lcc and v in lcc]
    n_lcc = len(lcc)
    b1_lcc = len(e_lcc) - n_lcc + 1 if n_lcc else 0
    deg = [len(adj[v]) for v in vset]
    leaf_fraction = sum(1 for v in lcc if len(adj[v]) <= 1) / n_lcc if n_lcc else float("nan")
    return {
        "n": int(n), "m": int(m), "components": int(c), "lcc_n": int(n_lcc),
        "lcc_ratio": float(n_lcc / n), "b1_full": int(b1_full),
        "rho_loop_full": float(b1_full / n), "b1_lcc": int(b1_lcc),
        "rho_loop_lcc": float(b1_lcc / n_lcc) if n_lcc else float("nan"),
        "degree_mean": float(np.mean(deg)) if deg else float("nan"),
        "degree_max": float(np.max(deg)) if deg else float("nan"),
        "leaf_fraction_lcc": float(leaf_fraction),
    }


def weighted_incidence(
    vertices: Sequence[int],
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> Dict[str, float]:
    values = {v: 0.0 for v in vertices}
    for u, v in edges:
        w = float(conductances.get(canonical_edge(u, v), 0.0))
        if u in values and v in values:
            values[u] += w
            values[v] += w
    arr = np.asarray(list(values.values()), dtype=float)
    return {
        "incidence_mean": float(np.mean(arr)) if arr.size else float("nan"),
        "incidence_max": float(np.max(arr)) if arr.size else float("nan"),
        "incidence_q95": float(np.quantile(arr, 0.95)) if arr.size else float("nan"),
    }


# ============================================================
# 2) Proxy cell-4 et activation relationnelle
# ============================================================

def generate_lattice4_proxy(
    mesh_level: int,
    patch_radius: float,
    base_spacing: float = 1.0,
) -> Tuple[List[int], List[Edge], Dict[int, Tuple[float, float]], Dict[int, Tuple[int, int]], float]:
    spacing = float(base_spacing) * (2.0 ** int(mesh_level))
    if spacing <= 0 or patch_radius <= 0:
        raise ValueError("spacing et patch_radius doivent être positifs.")
    mmax = int(math.floor(patch_radius / spacing))
    coords: List[Tuple[int, int]] = []
    for i in range(-mmax, mmax + 1):
        for j in range(-mmax, mmax + 1):
            x, y = i * spacing, j * spacing
            if x * x + y * y <= patch_radius * patch_radius + 1e-12:
                coords.append((i, j))
    id_of = {ij: idx for idx, ij in enumerate(coords)}
    coord_of = {idx: ij for ij, idx in id_of.items()}
    positions = {idx: (ij[0] * spacing, ij[1] * spacing) for idx, ij in coord_of.items()}
    vertices = list(range(len(coords)))
    edges: List[Edge] = []
    for i, j in coords:
        u = id_of[(i, j)]
        for di, dj in ((1, 0), (0, 1)):
            nb = (i + di, j + dj)
            if nb in id_of:
                edges.append(canonical_edge(u, id_of[nb]))
    return vertices, sorted(set(edges)), positions, coord_of, spacing


def interior_subgraph(
    vertices: Sequence[int],
    edges: Sequence[Edge],
    positions: Mapping[int, Tuple[float, float]],
    coord_of: Mapping[int, Tuple[int, int]],
    patch_radius: float,
    margin: float,
) -> Tuple[List[int], List[Edge], Dict[int, Tuple[float, float]], Dict[int, Tuple[int, int]]]:
    cutoff = patch_radius - margin
    keep = {v for v in vertices if math.hypot(*positions[v]) <= cutoff}
    v_out = sorted(keep)
    e_out = [canonical_edge(u, v) for u, v in edges if u in keep and v in keep]
    p_out = {v: positions[v] for v in v_out}
    c_out = {v: coord_of[v] for v in v_out}
    return v_out, sorted(set(e_out)), p_out, c_out


def activate_weighted_relations(
    geometric_edges: Sequence[Edge],
    vertices: Sequence[int],
    rng: np.random.Generator,
    delta_tau: float,
    cfg: C5LargePatchConfig,
) -> Tuple[List[Edge], Dict[Edge, float], Dict[str, float]]:
    p_keep = p_keep_from_resolution(
        delta_tau, cfg.tau_c, cfg.p_keep_at_tau_c,
        cfg.p_floor, cfg.p_ceil, cfg.kappa_keep,
    )
    lam = lambda_multiplicity_from_resolution(
        delta_tau, cfg.tau_c, cfg.lambda_uv,
        cfg.lambda_ir, cfg.kappa_lambda,
    )
    incidence = {v: 0.0 for v in vertices}
    active_edges: List[Edge] = []
    conductances: Dict[Edge, float] = {}
    proposed = clipped = rejected_by_cap = 0
    for idx in rng.permutation(len(geometric_edges)):
        u, v = geometric_edges[int(idx)]
        if rng.random() > p_keep:
            continue
        proposed += 1
        proposed_weight = float(1 + int(rng.poisson(lam)))
        remaining = min(cfg.incidence_cap - incidence[u], cfg.incidence_cap - incidence[v])
        if remaining < 1.0:
            rejected_by_cap += 1
            continue
        weight = min(proposed_weight, float(math.floor(remaining)))
        if weight < proposed_weight:
            clipped += 1
        key = canonical_edge(u, v)
        active_edges.append(key)
        conductances[key] = float(weight)
        incidence[u] += weight
        incidence[v] += weight
    return sorted(set(active_edges)), conductances, {
        "Z": resolution_coordinate(delta_tau, cfg.tP_cal),
        "p_keep": float(p_keep), "lambda_rel": float(lam),
        "proposed_edges": int(proposed), "active_edges": int(len(active_edges)),
        "weights_clipped": int(clipped),
        "edges_rejected_by_incidence_cap": int(rejected_by_cap),
    }


# ============================================================
# 3) Projection quotient par blocs — mode grand patch
# ============================================================

def make_block_partition(
    vertices: Sequence[int],
    coord_of: Mapping[int, Tuple[int, int]],
    block_size: int,
) -> Tuple[Dict[int, int], Dict[int, Tuple[int, int]], Dict[int, List[int]]]:
    if block_size < 2:
        raise ValueError("block_size doit être >= 2.")
    raw: Dict[Tuple[int, int], List[int]] = defaultdict(list)
    for u in vertices:
        i, j = coord_of[u]
        raw[(i // block_size, j // block_size)].append(u)
    block_coords = sorted(raw)
    block_id_of = {coord: bid for bid, coord in enumerate(block_coords)}
    block_coord_of = {bid: coord for coord, bid in block_id_of.items()}
    members: Dict[int, List[int]] = {}
    block_of_vertex: Dict[int, int] = {}
    for coord, verts in raw.items():
        bid = block_id_of[coord]
        members[bid] = sorted(verts)
        for u in verts:
            block_of_vertex[u] = bid
    return block_of_vertex, block_coord_of, members


def aggregate_block_projection(
    fine_vertices: Sequence[int],
    fine_edges: Sequence[Edge],
    fine_conductances: Mapping[Edge, float],
    block_of_vertex: Mapping[int, int],
    block_coord_of: Mapping[int, Tuple[int, int]],
) -> Tuple[List[int], List[Edge], Dict[Edge, float], Dict[str, float]]:
    coarse_vertices = sorted(set(block_of_vertex[v] for v in fine_vertices))
    coarse_c: Dict[Edge, float] = defaultdict(float)
    internal_weight = 0.0
    interblock_weight = 0.0
    nonlocal_edges = 0
    for u, v in fine_edges:
        w = float(fine_conductances[canonical_edge(u, v)])
        bu, bv = block_of_vertex[u], block_of_vertex[v]
        if bu == bv:
            internal_weight += w
            continue
        key = canonical_edge(bu, bv)
        coarse_c[key] += w
        interblock_weight += w
        x1, y1 = block_coord_of[bu]
        x2, y2 = block_coord_of[bv]
        if abs(x1 - x2) + abs(y1 - y2) != 1:
            nonlocal_edges += 1
    coarse_c = dict(coarse_c)
    coarse_edges = sorted(coarse_c)
    return coarse_vertices, coarse_edges, coarse_c, {
        "fine_internal_weight": float(internal_weight),
        "fine_interblock_weight": float(interblock_weight),
        "coarse_edges_nonlocal_count": int(nonlocal_edges),
        "projection_is_local": int(nonlocal_edges == 0),
    }


def quotient_energy_identity_error(
    fine_edges: Sequence[Edge],
    fine_conductances: Mapping[Edge, float],
    coarse_vertices: Sequence[int],
    coarse_edges: Sequence[Edge],
    coarse_conductances: Mapping[Edge, float],
    block_of_vertex: Mapping[int, int],
    rng: np.random.Generator,
    n_vectors: int,
) -> Dict[str, float]:
    index = {b: i for i, b in enumerate(coarse_vertices)}
    errors: List[float] = []
    for _ in range(max(1, int(n_vectors))):
        y = rng.normal(size=len(coarse_vertices))
        y -= np.mean(y)
        e_fine = 0.0
        for u, v in fine_edges:
            bu, bv = block_of_vertex[u], block_of_vertex[v]
            dy = y[index[bu]] - y[index[bv]]
            e_fine += float(fine_conductances[canonical_edge(u, v)]) * dy * dy
        e_coarse = 0.0
        for a, b in coarse_edges:
            dy = y[index[a]] - y[index[b]]
            e_coarse += float(coarse_conductances[canonical_edge(a, b)]) * dy * dy
        denom = max(abs(e_fine), 1e-14)
        errors.append(abs(e_fine - e_coarse) / denom)
    return {
        "quotient_energy_error_mean": float(np.mean(errors)),
        "quotient_energy_error_max": float(np.max(errors)),
    }


# ============================================================
# 4) Audit local échantillonné de Kron
# ============================================================

def reindex_graph(
    vertices: Sequence[int],
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> Tuple[List[int], List[Edge], Dict[Edge, float], Dict[int, int], Dict[int, int]]:
    old_to_new = {old: new for new, old in enumerate(sorted(vertices))}
    new_to_old = {new: old for old, new in old_to_new.items()}
    new_vertices = list(range(len(old_to_new)))
    new_edges: List[Edge] = []
    new_c: Dict[Edge, float] = {}
    for u, v in edges:
        if u not in old_to_new or v not in old_to_new:
            continue
        key_new = canonical_edge(old_to_new[u], old_to_new[v])
        new_edges.append(key_new)
        new_c[key_new] = float(conductances[canonical_edge(u, v)])
    return new_vertices, sorted(set(new_edges)), new_c, old_to_new, new_to_old


def laplacian_from_conductances(
    n_nodes: int,
    edges: Sequence[Edge],
    conductances: Mapping[Edge, float],
) -> csc_matrix:
    diag = np.zeros(n_nodes, dtype=float)
    rows: List[int] = []
    cols: List[int] = []
    data: List[float] = []
    for u, v in edges:
        w = float(conductances[canonical_edge(u, v)])
        diag[u] += w
        diag[v] += w
        rows.extend((u, v)); cols.extend((v, u)); data.extend((-w, -w))
    rows.extend(range(n_nodes)); cols.extend(range(n_nodes)); data.extend(diag.tolist())
    out = coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes)).tocsc()
    out.sum_duplicates()
    return out


def kron_reduction(
    laplacian: csc_matrix,
    keep_indices: Sequence[int],
    leak_regularization: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = laplacian.shape[0]
    keep = np.asarray(sorted(set(int(i) for i in keep_indices)), dtype=int)
    mask = np.ones(n, dtype=bool); mask[keep] = False
    eliminated = np.where(mask)[0]
    l_kk = laplacian[keep[:, None], keep]
    if eliminated.size == 0:
        return keep, eliminated, l_kk.toarray()
    l_ki = laplacian[keep[:, None], eliminated]
    l_ik = laplacian[eliminated[:, None], keep]
    l_ii = laplacian[eliminated[:, None], eliminated]
    diagonal = np.asarray(l_ii.diagonal(), dtype=float)
    positive = diagonal[diagonal > 0]
    scale = float(np.mean(positive)) if positive.size else 1.0
    eps = max(float(leak_regularization) * scale, 1e-15)
    lu = splu((l_ii + eps * identity(eliminated.size, format="csc")).tocsc())
    solution = lu.solve(l_ik.toarray())
    l_eff = l_kk.toarray() - (l_ki @ solution)
    l_eff = 0.5 * (l_eff + l_eff.T)
    return keep, eliminated, l_eff


def local_projection_from_kron(
    l_eff: np.ndarray,
    block_ids_in_eff_order: Sequence[int],
    block_coord_of: Mapping[int, Tuple[int, int]],
    weight_epsilon: float,
) -> Tuple[np.ndarray, Dict[str, float]]:
    total_weight = 0.0
    local_weight = 0.0
    k = l_eff.shape[0]
    l_local = np.zeros_like(l_eff)
    for a in range(k):
        for b in range(a + 1, k):
            w = float(-l_eff[a, b])
            if w <= weight_epsilon:
                continue
            total_weight += w
            ba, bb = block_ids_in_eff_order[a], block_ids_in_eff_order[b]
            xa, ya = block_coord_of[ba]
            xb, yb = block_coord_of[bb]
            if abs(xa - xb) + abs(ya - yb) == 1:
                local_weight += w
                l_local[a, a] += w; l_local[b, b] += w
                l_local[a, b] -= w; l_local[b, a] -= w
    nonlocal_weight = max(total_weight - local_weight, 0.0)
    frac = nonlocal_weight / total_weight if total_weight > 0 else float("nan")
    return l_local, {
        "kron_total_offdiag_weight": float(total_weight),
        "kron_local_retained_weight": float(local_weight),
        "kron_nonlocal_removed_weight": float(nonlocal_weight),
        "kron_nonlocal_fraction": float(frac),
    }


def energy_projection_error(
    l_eff: np.ndarray,
    l_local: np.ndarray,
    rng: np.random.Generator,
    n_vectors: int,
) -> Dict[str, float]:
    errors: List[float] = []
    n = l_eff.shape[0]
    for _ in range(max(1, int(n_vectors))):
        x = rng.normal(size=n); x -= np.mean(x)
        e_eff = float(x @ l_eff @ x)
        e_loc = float(x @ l_local @ x)
        errors.append(abs(e_eff - e_loc) / max(abs(e_eff), 1e-12))
    return {
        "kron_energy_error_mean": float(np.mean(errors)),
        "kron_energy_error_max": float(np.max(errors)),
    }


def sampled_local_kron_audit(
    fine_vertices: Sequence[int],
    fine_edges: Sequence[Edge],
    fine_conductances: Mapping[Edge, float],
    block_of_vertex: Mapping[int, int],
    block_coord_of: Mapping[int, Tuple[int, int]],
    block_members: Mapping[int, Sequence[int]],
    rng: np.random.Generator,
    cfg: C5LargePatchConfig,
) -> Dict[str, float]:
    if not cfg.run_local_kron_audit:
        return {"kron_audit_status": "disabled"}
    coord_to_block = {coord: bid for bid, coord in block_coord_of.items()}
    r = int(cfg.kron_audit_block_radius)
    candidates: List[int] = []
    for bid, (x, y) in block_coord_of.items():
        required = [(x + dx, y + dy) for dx in range(-r, r + 1) for dy in range(-r, r + 1)]
        if all(coord in coord_to_block for coord in required):
            candidates.append(bid)
    if not candidates:
        return {"kron_audit_status": "no_interior_windows"}
    chosen = rng.choice(candidates, size=min(cfg.kron_audit_samples, len(candidates)), replace=False)
    fine_edge_set = list(fine_edges)
    fractions: List[float] = []
    energy_maxes: List[float] = []
    windows_completed = 0
    for center in chosen:
        cx, cy = block_coord_of[int(center)]
        block_ids = {
            coord_to_block[(cx + dx, cy + dy)]
            for dx in range(-r, r + 1)
            for dy in range(-r, r + 1)
        }
        verts = sorted({u for bid in block_ids for u in block_members[bid]})
        if len(verts) > cfg.kron_audit_max_nodes:
            continue
        vset = set(verts)
        edges = [e for e in fine_edge_set if e[0] in vset and e[1] in vset]
        if len(verts) < 4 or not edges:
            continue
        new_v, new_e, new_c, old_to_new, new_to_old = reindex_graph(verts, edges, fine_conductances)
        anchors_old = [min(block_members[bid]) for bid in sorted(block_ids)]
        anchor_pairs = sorted(
            (old_to_new[a], block_of_vertex[a])
            for a in anchors_old
            if a in old_to_new
        )
        anchors_new = [idx for idx, _ in anchor_pairs]
        block_ids_order = [bid for _, bid in anchor_pairs]
        if len(anchors_new) < 2:
            continue
        lap = laplacian_from_conductances(len(new_v), new_e, new_c)
        try:
            _, _, l_eff = kron_reduction(lap, anchors_new, cfg.kron_leak_regularization)
        except Exception:
            continue
        l_local, wdiag = local_projection_from_kron(
            l_eff, block_ids_order, block_coord_of, cfg.weight_epsilon
        )
        ediag = energy_projection_error(
            l_eff, l_local, rng, cfg.kron_energy_test_vectors
        )
        if np.isfinite(wdiag["kron_nonlocal_fraction"]):
            fractions.append(wdiag["kron_nonlocal_fraction"])
        if np.isfinite(ediag["kron_energy_error_max"]):
            energy_maxes.append(ediag["kron_energy_error_max"])
        windows_completed += 1
    if windows_completed == 0:
        return {"kron_audit_status": "no_completed_windows", "kron_windows_completed": 0}
    frac_mean = float(np.mean(fractions)) if fractions else float("nan")
    frac_max = float(np.max(fractions)) if fractions else float("nan")
    energy_mean = float(np.mean(energy_maxes)) if energy_maxes else float("nan")
    energy_max = float(np.max(energy_maxes)) if energy_maxes else float("nan")
    audit_pass = int(
        np.isfinite(frac_max) and np.isfinite(energy_max)
        and frac_max <= cfg.kron_nonlocal_fraction_max
        and energy_max <= cfg.kron_energy_error_max
    )
    return {
        "kron_audit_status": "completed",
        "kron_windows_completed": int(windows_completed),
        "kron_nonlocal_fraction_mean": frac_mean,
        "kron_nonlocal_fraction_max": frac_max,
        "kron_energy_error_window_mean": energy_mean,
        "kron_energy_error_window_max": energy_max,
        "kron_audit_pass": audit_pass,
    }


# ============================================================
# 5) Verdicts et synthèse
# ============================================================

def classify_realization(row: Mapping[str, object], cfg: C5LargePatchConfig) -> str:
    if int(row["fine_quality_pass"]) == 0:
        return "inconclusive_fragmented"
    if int(row["projected_quality_pass"]) == 0:
        return "inconclusive_fragmented_after_projection"
    if int(row["incidence_pass"]) == 0:
        return "fail_incidence"
    if int(row["quotient_locality_pass"]) == 0:
        return "fail_nonlocal_quotient_projection"
    if int(row["energy_identity_pass"]) == 0:
        return "fail_quotient_energy_identity"
    rho_f = float(row["rho_loop_lcc"])
    rho_p = float(row["rho_loop_lcc_projected"])
    leaf_f = float(row["leaf_fraction_lcc"])
    leaf_p = float(row["leaf_fraction_lcc_projected"])
    if (
        rho_f >= cfg.rho_loop_min and rho_p >= cfg.rho_loop_min
        and leaf_f <= cfg.leaf_fraction_max and leaf_p <= cfg.leaf_fraction_max
    ):
        return "pass_connected_non_tree_like"
    if (
        rho_f < cfg.rho_loop_min and rho_p < cfg.rho_loop_min
        and leaf_f > cfg.leaf_fraction_max and leaf_p > cfg.leaf_fraction_max
    ):
        return "candidate_tree_like"
    return "inconclusive_mixed_cycle_diagnostics"


def relative_range(values: Sequence[float]) -> float:
    arr = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size < 2:
        return float("nan")
    return float((np.max(arr) - np.min(arr)) / max(abs(float(np.mean(arr))), 1e-12))


def summarize_results(df: pd.DataFrame, cfg: C5LargePatchConfig) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for (delta_tau, mesh_level), group in df.groupby(["delta_tau", "mesh_level"], dropna=False):
        group = group.sort_values(["patch_radius", "realization"])
        by_radius = group.groupby("patch_radius", as_index=False).agg(
            fine_quality_rate=("fine_quality_pass", "mean"),
            projected_quality_rate=("projected_quality_pass", "mean"),
            incidence_pass_rate=("incidence_pass", "mean"),
            quotient_locality_pass_rate=("quotient_locality_pass", "mean"),
            energy_identity_pass_rate=("energy_identity_pass", "mean"),
            rho_fine_mean=("rho_loop_lcc", "mean"),
            rho_projected_mean=("rho_loop_lcc_projected", "mean"),
            lcc_fine_mean=("lcc_ratio", "mean"),
            lcc_projected_mean=("lcc_ratio_projected", "mean"),
        )
        admissible = by_radius[
            (by_radius["fine_quality_rate"] >= 0.75)
            & (by_radius["projected_quality_rate"] >= 0.75)
            & (by_radius["incidence_pass_rate"] >= 1.0)
            & (by_radius["quotient_locality_pass_rate"] >= 1.0)
            & (by_radius["energy_identity_pass_rate"] >= 1.0)
        ]
        last_radius = float(group["patch_radius"].max())
        last = group[group["patch_radius"] == last_radius]
        counts = last["verdict"].value_counts(dropna=False).to_dict()
        dominant = str(last["verdict"].mode().iloc[0]) if not last.empty else "inconclusive_no_data"
        if len(admissible) >= cfg.min_patch_points:
            rr_f = relative_range(admissible["rho_fine_mean"].tolist())
            rr_p = relative_range(admissible["rho_projected_mean"].tolist())
            stable = int(rr_f <= cfg.patch_relative_range_max and rr_p <= cfg.patch_relative_range_max)
        else:
            rr_f = rr_p = float("nan")
            stable = 0
        pass_rate_last = float(np.mean(last["verdict"] == "pass_connected_non_tree_like")) if len(last) else 0.0
        if dominant == "pass_connected_non_tree_like" and pass_rate_last >= 0.75 and stable:
            summary_verdict = "pass_large_patch_non_tree_like"
        elif dominant == "pass_connected_non_tree_like":
            summary_verdict = "provisional_pass_large_patch_stability_not_established"
        elif dominant.startswith("inconclusive"):
            summary_verdict = dominant
        else:
            summary_verdict = dominant
        rows.append({
            "delta_tau": float(delta_tau),
            "Z": resolution_coordinate(float(delta_tau), cfg.tP_cal),
            "mesh_level": int(mesh_level),
            "patch_radius_max": last_radius,
            "n_fine_mean_at_max_radius": float(last["n_fine"].mean()),
            "n_projected_mean_at_max_radius": float(last["n_projected"].mean()),
            "n_realizations_at_max_radius": int(len(last)),
            "pass_rate_at_max_radius": pass_rate_last,
            "dominant_verdict_at_max_radius": dominant,
            "verdict_counts_at_max_radius": json.dumps(counts, sort_keys=True),
            "admissible_patch_points": int(len(admissible)),
            "rho_relative_range_fine": rr_f,
            "rho_relative_range_projected": rr_p,
            "large_patch_stability_pass": stable,
            "summary_verdict": summary_verdict,
        })
    return pd.DataFrame(rows).sort_values(["delta_tau", "mesh_level"]).reset_index(drop=True)


# ============================================================
# 6) Exécution avec checkpoint/reprise
# ============================================================

def _completed_keys(df: pd.DataFrame) -> set:
    if df.empty:
        return set()
    cols = ["delta_tau", "patch_radius", "mesh_level", "realization"]
    return set(tuple(row) for row in df[cols].itertuples(index=False, name=None))


def run_c5_large_patch(
    cfg: C5LargePatchConfig,
    output_dir: str | Path = ".",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / cfg.results_csv
    summary_path = output_dir / cfg.summary_csv
    metadata_path = output_dir / cfg.metadata_json

    if cfg.resume and results_path.exists():
        existing = pd.read_csv(results_path)
    else:
        existing = pd.DataFrame()
    done = _completed_keys(existing)
    new_rows: List[Dict[str, object]] = []

    total = len(cfg.delta_tau_values) * len(cfg.patch_radii) * len(cfg.mesh_levels) * cfg.n_realizations
    completed_before = len(done)
    counter = completed_before
    print(f"Configurations prévues: {total}; déjà présentes: {completed_before}")

    for delta_tau in cfg.delta_tau_values:
        for patch_radius in cfg.patch_radii:
            for mesh_level in cfg.mesh_levels:
                for realization in range(cfg.n_realizations):
                    key = (float(delta_tau), float(patch_radius), int(mesh_level), int(realization))
                    if key in done:
                        continue
                    counter += 1
                    local_seed = (
                        cfg.seed + int(round(delta_tau * 1e6)) + 100003 * mesh_level
                        + 1009 * int(round(patch_radius)) + 7919 * realization
                    )
                    rng = np.random.default_rng(local_seed)
                    vertices, geo_edges, positions, coord_of, spacing = generate_lattice4_proxy(
                        mesh_level, patch_radius, cfg.base_mesh_spacing
                    )
                    vertices, geo_edges, positions, coord_of = interior_subgraph(
                        vertices, geo_edges, positions, coord_of,
                        patch_radius, cfg.margin_factor * spacing,
                    )
                    active_edges, conductances, activation = activate_weighted_relations(
                        geo_edges, vertices, rng, delta_tau, cfg
                    )
                    fine_m = graph_metrics(vertices, active_edges)
                    inc = weighted_incidence(vertices, active_edges, conductances)
                    block_of, block_coord, block_members = make_block_partition(
                        vertices, coord_of, cfg.block_size
                    )
                    coarse_v, coarse_e, coarse_c, proj = aggregate_block_projection(
                        vertices, active_edges, conductances, block_of, block_coord
                    )
                    coarse_m = graph_metrics(coarse_v, coarse_e)
                    energy = quotient_energy_identity_error(
                        active_edges, conductances, coarse_v, coarse_e, coarse_c,
                        block_of, rng, cfg.energy_test_vectors,
                    )
                    fine_quality = int(
                        fine_m["n"] >= cfg.n_min_fine and fine_m["lcc_ratio"] >= cfg.chi_min
                    )
                    projected_quality = int(
                        coarse_m["n"] >= cfg.n_min_projected and coarse_m["lcc_ratio"] >= cfg.chi_min
                    )
                    row: Dict[str, object] = {
                        "delta_tau": float(delta_tau),
                        "Z": activation["Z"],
                        "ifs_depth_fixed": int(cfg.ifs_depth_fixed),
                        "mesh_level": int(mesh_level),
                        "mesh_spacing": float(spacing),
                        "patch_radius": float(patch_radius),
                        "realization": int(realization),
                        "seed_local": int(local_seed),
                        "p_keep": activation["p_keep"],
                        "lambda_rel": activation["lambda_rel"],
                        "n_fine": fine_m["n"],
                        "m_fine": fine_m["m"],
                        "lcc_ratio": fine_m["lcc_ratio"],
                        "rho_loop_lcc": fine_m["rho_loop_lcc"],
                        "leaf_fraction_lcc": fine_m["leaf_fraction_lcc"],
                        "degree_max": fine_m["degree_max"],
                        "incidence_max": inc["incidence_max"],
                        "incidence_q95": inc["incidence_q95"],
                        "fine_quality_pass": fine_quality,
                        "incidence_pass": int(inc["incidence_max"] <= cfg.incidence_cap + 1e-12),
                        "n_projected": coarse_m["n"],
                        "m_projected": coarse_m["m"],
                        "lcc_ratio_projected": coarse_m["lcc_ratio"],
                        "rho_loop_lcc_projected": coarse_m["rho_loop_lcc"],
                        "leaf_fraction_lcc_projected": coarse_m["leaf_fraction_lcc"],
                        "degree_max_projected": coarse_m["degree_max"],
                        "projected_quality_pass": projected_quality,
                        "coarse_edges_nonlocal_count": proj["coarse_edges_nonlocal_count"],
                        "quotient_locality_pass": int(proj["projection_is_local"] == 1),
                        "quotient_energy_error_mean": energy["quotient_energy_error_mean"],
                        "quotient_energy_error_max": energy["quotient_energy_error_max"],
                        "energy_identity_pass": int(energy["quotient_energy_error_max"] <= cfg.energy_identity_tol),
                        "weights_clipped": activation["weights_clipped"],
                        "edges_rejected_by_incidence_cap": activation["edges_rejected_by_incidence_cap"],
                    }
                    # Kron n'est qu'un audit local, sans conditionner le verdict principal.
                    if fine_quality and projected_quality:
                        row.update(sampled_local_kron_audit(
                            vertices, active_edges, conductances, block_of,
                            block_coord, block_members, rng, cfg,
                        ))
                    else:
                        row.update({"kron_audit_status": "skipped_fragmented"})
                    row["verdict"] = classify_realization(row, cfg)
                    new_rows.append(row)
                    print(
                        f"[{counter}/{total}] tau={delta_tau:g}, R={patch_radius:g}, "
                        f"r={realization}: {row['verdict']}"
                    )

                # Checkpoint après chaque triplet tau/R/mesh.
                if new_rows:
                    current = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
                    current = current.drop_duplicates(
                        subset=["delta_tau", "patch_radius", "mesh_level", "realization"],
                        keep="last",
                    ).sort_values(["delta_tau", "patch_radius", "mesh_level", "realization"])
                    current.to_csv(results_path, index=False)

    if new_rows:
        results = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    else:
        results = existing.copy()
    results = results.drop_duplicates(
        subset=["delta_tau", "patch_radius", "mesh_level", "realization"], keep="last"
    ).sort_values(["delta_tau", "patch_radius", "mesh_level", "realization"]).reset_index(drop=True)
    summary = summarize_results(results, cfg)
    results.to_csv(results_path, index=False)
    summary.to_csv(summary_path, index=False)
    metadata = {
        "test_name": "C5_large_patch_non_arborescence_cell4_projection_v5",
        "status": "completed",
        "conceptual_guardrails": {
            "ifs_depth_fixed": cfg.ifs_depth_fixed,
            "coarse_graining_is_projection_not_ifs_iteration": True,
            "global_kron_reduction_used": False,
            "primary_projection": "connected_block_quotient_with_conductance_sums",
            "kron_role": "optional_sampled_local_audit_only",
            "planck_cells_postulated": False,
            "dsi_used": False,
            "uv_fragmentation_interpretation": "inconclusive_for_global_topology",
        },
        "config": asdict(cfg),
        "versions": {
            "python": sys.version,
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
        },
        "n_rows": int(len(results)),
        "summary_verdict_counts": summary["summary_verdict"].value_counts().to_dict(),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Exports générés:")
    print(" -", results_path)
    print(" -", summary_path)
    print(" -", metadata_path)
    return results, summary


# ============================================================
# 7) Autotests
# ============================================================

def run_autotests() -> None:
    cfg = config_for_mode("SMOKE")
    assert abs(p_keep_from_resolution(
        cfg.tau_c, cfg.tau_c, cfg.p_keep_at_tau_c,
        cfg.p_floor, cfg.p_ceil, cfg.kappa_keep,
    ) - cfg.p_keep_at_tau_c) < 1e-12
    rng = np.random.default_rng(7)
    v, e, pos, coord, spacing = generate_lattice4_proxy(0, 10.0, 1.0)
    v, e, pos, coord = interior_subgraph(v, e, pos, coord, 10.0, 2.0)
    active, c, _ = activate_weighted_relations(e, v, rng, 1.0, cfg)
    block_of, block_coord, members = make_block_partition(v, coord, 2)
    cv, ce, cc, proj = aggregate_block_projection(v, active, c, block_of, block_coord)
    assert proj["coarse_edges_nonlocal_count"] == 0
    energy = quotient_energy_identity_error(active, c, cv, ce, cc, block_of, rng, 4)
    assert energy["quotient_energy_error_max"] < 1e-12
    inc = weighted_incidence(v, active, c)
    assert inc["incidence_max"] <= cfg.incidence_cap + 1e-12
    gm = graph_metrics(v, active)
    assert gm["degree_max"] <= 4
    assert cfg.ifs_depth_fixed == 0
    print("Autotests C.5 V5 grand patch: OK")

## Execution

The notebook is configured by default in `LARGE_PATCH` mode.

For a quick check, replace:

```python
RUN_MODE = "LARGE_PATCH"
```

with:

```python
RUN_MODE = "SMOKE"
```

To preserve checkpoints between two Colab sessions, place `OUTPUT_DIR` in Google Drive after mounting Drive. Resume is enabled by default in the large patch configuration.

In [ ]:
from pathlib import Path

RUN_MODE = V5_RUN_MODE
OUTPUT_DIR = MASTER_OUTPUT_DIR / "C5_large_patch_v5"

if RUN_V5:
    run_autotests()
    cfg = config_for_mode(RUN_MODE)
    results_v5, summary_v5 = run_c5_large_patch(cfg, output_dir=OUTPUT_DIR)
    display(summary_v5)
else:
    results_v5, summary_v5 = pd.DataFrame(), pd.DataFrame()

Autotests C.5 V5 grand patch: OK
Configurations prévues: 128; déjà présentes: 0
[1/128] tau=1, R=32, r=0: pass_connected_non_tree_like
[2/128] tau=1, R=32, r=1: pass_connected_non_tree_like
[3/128] tau=1, R=32, r=2: pass_connected_non_tree_like
[4/128] tau=1, R=32, r=3: pass_connected_non_tree_like
[5/128] tau=1, R=32, r=4: pass_connected_non_tree_like
[6/128] tau=1, R=32, r=5: pass_connected_non_tree_like
[7/128] tau=1, R=32, r=6: pass_connected_non_tree_like
[8/128] tau=1, R=32, r=7: pass_connected_non_tree_like
[9/128] tau=1, R=48, r=0: pass_connected_non_tree_like
[10/128] tau=1, R=48, r=1: pass_connected_non_tree_like
[11/128] tau=1, R=48, r=2: pass_connected_non_tree_like
[12/128] tau=1, R=48, r=3: pass_connected_non_tree_like
[13/128] tau=1, R=48, r=4: pass_connected_non_tree_like
[14/128] tau=1, R=48, r=5: pass_connected_non_tree_like
[15/128] tau=1, R=48, r=6: pass_connected_non_tree_like
[16/128] tau=1, R=48, r=7: pass_connected_non_tree_like
[17/128] tau=1, R=64, r=0: pass_c

,delta_tau,Z,mesh_level,patch_radius_max,n_fine_mean_at_max_radius,n_projected_mean_at_max_radius,n_realizations_at_max_radius,pass_rate_at_max_radius,dominant_verdict_at_max_radius,verdict_counts_at_max_radius,admissible_patch_points,rho_relative_range_fine,rho_relative_range_projected,large_patch_stability_pass,summary_verdict
0,0.04,-3.218876,0,96.0,27145.0,6880.0,8,0.0,inconclusive_fragmented,"{""inconclusive_fragmented"": 8}",0,NaN,NaN,0,inconclusive_fragmented
1,0.08,-2.525729,0,96.0,27145.0,6880.0,8,0.0,inconclusive_fragmented,"{""inconclusive_fragmented"": 8}",0,NaN,NaN,0,inconclusive_fragmented
2,0.20,-1.609438,0,96.0,27145.0,6880.0,8,1.0,pass_connected_non_tree_like,"{""pass_connected_non_tree_like"": 8}",4,0.057345,0.079984,1,pass_large_patch_non_tree_like
3,1.00,0.000000,0,96.0,27145.0,6880.0,8,1.0,pass_connected_non_tree_like,"{""pass_connected_non_tree_like"": 8}",4,0.033875,0.065610,1,pass_large_patch_non_tree_like


## Reading Results

The main verdict is based on:

1. macroscopic connectivity of the fine graph;
2. connectivity of the quotient graph;
3. cycle density before and after projection;
4. leaf fraction;
5. compliance with the incidence bound;
6. exact locality of the quotient projection;
7. energetic identity for block-constant functions;
8. stability of diagnostics as patch radius increases.

The main outputs are:

- `C5_results_large_patch_v5.csv`;
- `C5_summary_large_patch_v5.csv`;
- `C5_run_metadata_large_patch_v5.json`.

The local Kron audit separately measures the proportion of non-local couplings that a Schur complement would have produced on bounded windows. It no longer defines the primary coarse-graining.


# V4 / V5 Synthesis

At the end of execution:

- V4 outputs are directly in `C5_V4_V5_outputs/`;
- V5 outputs are in `C5_V4_V5_outputs/C5_large_patch_v5/`.

The recommended comparison is on:

\[\chi_{\mathrm{LCC}},
\qquad
\rho_{\mathrm{loop}},
\qquad
m_{\max},
\]

before and after projection.

V4 also provides a global audit of non-local leakage produced by Kron on modest sizes.

V5 provides stability at larger sizes through a local, energetically exact quotient projection for block-constant functions.

In case of divergence between the two versions:

1. first check finite-size effects;
2. examine the non-local fraction measured by V4;
3. check connectivity and incidence thresholds;
4. never convert a fragmented regime into a positive or negative topological verdict.